# BeeWatch — Sensor Autoencoder Training
**Bagian:** Model Sensor  
**Framework:** TensorFlow / Keras  
**Dataset:** Kaggle `annajyang/beehive-sounds` (`all_data_updated.csv`)  
**Fitur input:** `hive temp`, `hive humidity`, `hive pressure` (3 fitur, tanpa hive weight)

---
### Alur Notebook
1. Setup & import
2. Load dataset
3. Eksplorasi data sensor
4. Persiapan data (split → normalisasi)
5. Bangun & latih Sensor Autoencoder (Keras)
6. Visualisasi training
7. Hitung reconstruction error & threshold
8. Evaluasi & visualisasi lanjutan
9. Simpan artefak
10. Verifikasi inferensi


## 1. Setup & Import

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import json
import os

import tensorflow as tf
from tensorflow import keras
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA

np.random.seed(42)
tf.random.set_seed(42)
os.makedirs('models', exist_ok=True)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)

print(f"TensorFlow  : {tf.__version__}")
print(f"GPU tersedia: {len(tf.config.list_physical_devices('GPU')) > 0}")


## 2. Load Dataset

In [ ]:
df = pd.read_csv('/kaggle/input/beehive-sounds/all_data_updated.csv')
print(f"Shape dataset : {df.shape}")
print(f"Kolom         : {df.columns.tolist()}")
print()
df.head()


## 3. Eksplorasi Data Sensor

In [ ]:
SENSOR_FEATURES = ['hive temp', 'hive humidity', 'hive pressure']
INPUT_DIM       = len(SENSOR_FEATURES)

print("=== Missing Values ===")
print(df[SENSOR_FEATURES].isnull().sum())
print()
print("=== Statistik Deskriptif ===")
print(df[SENSOR_FEATURES].describe().round(3))


In [ ]:
# Distribusi tiap fitur sensor
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
colors = ['steelblue', 'seagreen', 'tomato']
labels = ['Suhu (°C)', 'Kelembaban (%)', 'Tekanan (hPa)']

for i, (feat, label, color) in enumerate(zip(SENSOR_FEATURES, labels, colors)):
    axes[i].hist(df[feat], bins=40, color=color, alpha=0.8, edgecolor='white')
    axes[i].set_title(f'Distribusi: {label}', fontsize=12)
    axes[i].set_xlabel(label)
    axes[i].set_ylabel('Frekuensi')
    axes[i].grid(True, alpha=0.3)

plt.suptitle('Distribusi Fitur Sensor', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('sensor_distribution.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Heatmap korelasi antar fitur sensor
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

corr = df[SENSOR_FEATURES].corr()
sns.heatmap(corr, annot=True, fmt='.3f', cmap='coolwarm',
            center=0, ax=axes[0], square=True,
            xticklabels=['Suhu', 'Kelembaban', 'Tekanan'],
            yticklabels=['Suhu', 'Kelembaban', 'Tekanan'])
axes[0].set_title('Korelasi Antar Fitur Sensor')

# Scatter matrix
pd.plotting.scatter_matrix(
    df[SENSOR_FEATURES], ax=axes[1],
    alpha=0.3, figsize=(6, 5), diagonal='hist',
    color='steelblue'
)
axes[1].set_title('Scatter Matrix')

plt.suptitle('Analisis Korelasi Sensor', fontsize=14)
plt.tight_layout()
plt.savefig('sensor_correlation.png', dpi=150, bbox_inches='tight')
plt.show()


## 4. Persiapan Data

In [ ]:
X_sensor = df[SENSOR_FEATURES].values
print(f"Shape data sensor: {X_sensor.shape}")

# Split DULU, baru normalisasi
X_train_raw, X_val_raw = train_test_split(
    X_sensor, test_size=0.15, random_state=42
)
print(f"Training samples  : {X_train_raw.shape[0]}")
print(f"Validation samples: {X_val_raw.shape[0]}")

sensor_scaler = StandardScaler()
X_train = sensor_scaler.fit_transform(X_train_raw)
X_val   = sensor_scaler.transform(X_val_raw)

print(f"\nSetelah normalisasi (train):")
print(f"  Mean per fitur : {X_train.mean(axis=0).round(4)}  (~0)")
print(f"  Std  per fitur : {X_train.std(axis=0).round(4)}   (~1)")


## 5. Arsitektur Model

In [ ]:
ENCODING_DIM = 2

def build_sensor_autoencoder(input_dim: int, encoding_dim: int) -> keras.Model:
    inputs = keras.Input(shape=(input_dim,), name='sensor_input')

    # Encoder
    x = keras.layers.Dense(16, activation='relu', name='enc_dense1')(inputs)
    x = keras.layers.BatchNormalization(name='enc_bn1')(x)
    encoded = keras.layers.Dense(encoding_dim, activation='relu', name='bottleneck')(x)

    # Decoder
    x = keras.layers.Dense(16, activation='relu', name='dec_dense1')(encoded)
    x = keras.layers.BatchNormalization(name='dec_bn1')(x)
    decoded = keras.layers.Dense(input_dim, activation='linear', name='sensor_output')(x)

    return keras.Model(inputs=inputs, outputs=decoded, name='SensorAutoencoder')


sensor_model = build_sensor_autoencoder(INPUT_DIM, ENCODING_DIM)
sensor_model.summary()


## 6. Training

In [ ]:
sensor_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='mse',
    metrics=['mae']
)

cbs = [
    keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=15,
        restore_best_weights=True, verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5,
        patience=7, min_lr=1e-6, verbose=1
    )
]

history = sensor_model.fit(
    X_train, X_train,
    validation_data=(X_val, X_val),
    epochs=200,
    batch_size=32,
    callbacks=cbs,
    verbose=1
)

final_epoch = len(history.history['loss'])
print(f"\nTraining selesai di epoch : {final_epoch}")
print(f"Final train loss          : {history.history['loss'][-1]:.6f}")
print(f"Final val loss            : {history.history['val_loss'][-1]:.6f}")
print(f"Best val loss             : {min(history.history['val_loss']):.6f}")


## 7. Visualisasi Training

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

best_ep = int(np.argmin(history.history['val_loss']))

axes[0].plot(history.history['loss'],     label='Train Loss', color='steelblue', linewidth=2)
axes[0].plot(history.history['val_loss'], label='Val Loss',   color='tomato',    linewidth=2)
axes[0].axvline(best_ep, color='gray', linestyle=':', alpha=0.7,
                label=f'Best epoch {best_ep+1}')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('MSE Loss')
axes[0].set_title('Training & Validation Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history.history['mae'],     label='Train MAE', color='steelblue', linewidth=2)
axes[1].plot(history.history['val_mae'], label='Val MAE',   color='tomato',    linewidth=2)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('MAE')
axes[1].set_title('Training & Validation MAE')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('Sensor Autoencoder — Kurva Training', fontsize=14)
plt.tight_layout()
plt.savefig('sensor_training_loss.png', dpi=150, bbox_inches='tight')
plt.show()


## 8. Reconstruction Error & Threshold

In [ ]:
X_train_recon = sensor_model.predict(X_train, verbose=0)
train_errors  = np.mean((X_train - X_train_recon) ** 2, axis=1)

X_val_recon = sensor_model.predict(X_val, verbose=0)
val_errors  = np.mean((X_val - X_val_recon) ** 2, axis=1)

print("=== Reconstruction Error (Training) ===")
print(f"  Mean : {train_errors.mean():.6f}")
print(f"  Std  : {train_errors.std():.6f}")
print(f"  P90  : {np.percentile(train_errors, 90):.6f}")
print(f"  P95  : {np.percentile(train_errors, 95):.6f}")
print(f"  P99  : {np.percentile(train_errors, 99):.6f}")

print("\n=== Reconstruction Error (Validation) ===")
print(f"  Mean : {val_errors.mean():.6f}")
print(f"  P95  : {np.percentile(val_errors, 95):.6f}")

sensor_threshold = float(np.percentile(train_errors, 95))
print(f"\n>>> Threshold P95 : {sensor_threshold:.6f}")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(train_errors, bins=50, alpha=0.7, color='steelblue',
             edgecolor='white', label='Train')
axes[0].hist(val_errors,   bins=50, alpha=0.5, color='seagreen',
             edgecolor='white', label='Val')
axes[0].axvline(sensor_threshold, color='red', linestyle='--',
                linewidth=2, label=f'Threshold P95 = {sensor_threshold:.4f}')
axes[0].set_xlabel('Reconstruction Error (MSE)')
axes[0].set_ylabel('Frekuensi')
axes[0].set_title('Distribusi Reconstruction Error')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].scatter(range(len(train_errors)), np.sort(train_errors),
                s=5, alpha=0.5, color='steelblue')
axes[1].axhline(sensor_threshold, color='red', linestyle='--',
                linewidth=2, label=f'Threshold = {sensor_threshold:.4f}')
axes[1].set_xlabel('Sampel (diurutkan)')
axes[1].set_ylabel('Reconstruction Error')
axes[1].set_title('Error Terurut — Train')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('Sensor Autoencoder — Analisis Reconstruction Error', fontsize=14)
plt.tight_layout()
plt.savefig('sensor_error_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Anomaly rate train : {(train_errors > sensor_threshold).mean()*100:.1f}%")
print(f"Anomaly rate val   : {(val_errors   > sensor_threshold).mean()*100:.1f}%")


## 9. Evaluasi Lanjutan

In [ ]:
# Sensitivitas threshold
print("Analisis Sensitivitas Threshold:")
print(f"{'Persentil':>10} {'Threshold':>12} {'Anomali (n)':>12} {'Anomali (%)':>12}")
print('-' * 50)
for pct in [90, 92, 95, 97, 98, 99]:
    thr_val = float(np.percentile(train_errors, pct))
    n_anom  = int((train_errors > thr_val).sum())
    marker  = ' ← dipilih' if pct == 95 else ''
    print(f"{pct:>10} {thr_val:>12.6f} {n_anom:>12} "
          f"{n_anom/len(train_errors)*100:>11.1f}%{marker}")


In [ ]:
# Boxplot train vs val
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].boxplot(
    [train_errors, val_errors],
    labels=['Train', 'Val'],
    patch_artist=True,
    boxprops=dict(facecolor='lightblue', color='steelblue'),
    medianprops=dict(color='red', linewidth=2)
)
axes[0].axhline(sensor_threshold, color='red', linestyle='--',
                linewidth=1.5, label=f'Threshold = {sensor_threshold:.4f}')
axes[0].set_ylabel('Reconstruction Error')
axes[0].set_title('Error Distribution: Train vs Val')
axes[0].legend()
axes[0].grid(True, alpha=0.3, axis='y')

# Bottleneck 2D
encoder_model   = keras.Model(inputs=sensor_model.input,
                               outputs=sensor_model.get_layer('bottleneck').output)
X_all_norm      = sensor_scaler.transform(X_sensor)
bottleneck_repr = encoder_model.predict(X_all_norm, verbose=0)
all_errors_all  = np.mean(
    (X_all_norm - sensor_model.predict(X_all_norm, verbose=0)) ** 2, axis=1
)

sc = axes[1].scatter(bottleneck_repr[:, 0], bottleneck_repr[:, 1],
                     c=all_errors_all, cmap='YlOrRd', alpha=0.6, s=15)
plt.colorbar(sc, ax=axes[1], label='Reconstruction Error')
axes[1].set_xlabel('Dimensi Bottleneck 1')
axes[1].set_ylabel('Dimensi Bottleneck 2')
axes[1].set_title('Representasi Bottleneck (2D)\nWarna = Reconstruction Error')
axes[1].grid(True, alpha=0.3)

plt.suptitle('Sensor Autoencoder — Evaluasi', fontsize=14)
plt.tight_layout()
plt.savefig('sensor_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Ringkasan evaluasi
print('=' * 55)
print(f"{'RINGKASAN EVALUASI SENSOR AUTOENCODER':^55}")
print('=' * 55)
print(f"{'Metrik':<35} {'Nilai':>18}")
print('-' * 55)
print(f"{'Training samples':<35} {X_train.shape[0]:>18,}")
print(f"{'Validation samples':<35} {X_val.shape[0]:>18,}")
print(f"{'Feature dimension':<35} {INPUT_DIM:>18}")
print(f"{'Encoding dimension':<35} {ENCODING_DIM:>18}")
print(f"{'Epochs trained':<35} {final_epoch:>18}")
print(f"{'Best val loss':<35} {min(history.history['val_loss']):>18.6f}")
print(f"{'Train error mean':<35} {train_errors.mean():>18.6f}")
print(f"{'Train error std':<35} {train_errors.std():>18.6f}")
print(f"{'Threshold (P95)':<35} {sensor_threshold:>18.6f}")
print(f"{'Anomaly rate train':<35} {(train_errors>sensor_threshold).mean()*100:>17.1f}%")
print(f"{'Anomaly rate val':<35} {(val_errors>sensor_threshold).mean()*100:>17.1f}%")


## 10. Simpan Artefak

In [ ]:
sensor_model.save('models/sensor_autoencoder.keras')
print("✅ Model     : models/sensor_autoencoder.keras")

with open('models/sensor_scaler.pkl', 'wb') as f:
    pickle.dump(sensor_scaler, f)
print("✅ Scaler    : models/sensor_scaler.pkl")

np.save('models/sensor_threshold.npy', np.array(sensor_threshold, dtype=np.float32))
print("✅ Threshold : models/sensor_threshold.npy")

model_info = {
    'input_features'      : SENSOR_FEATURES,
    'input_dim'           : INPUT_DIM,
    'encoding_dim'        : ENCODING_DIM,
    'threshold_percentile': 95,
    'threshold_value'     : sensor_threshold,
    'training_samples'    : int(X_train.shape[0]),
    'val_samples'         : int(X_val.shape[0]),
    'final_train_loss'    : float(history.history['loss'][-1]),
    'final_val_loss'      : float(history.history['val_loss'][-1]),
    'epochs_trained'      : final_epoch,
    'tensorflow_version'  : tf.__version__,
    'scaler_mean'         : sensor_scaler.mean_.tolist(),
    'scaler_std'          : sensor_scaler.scale_.tolist(),
}
with open('models/sensor_model_info.json', 'w') as f:
    json.dump(model_info, f, indent=2)
print("✅ Info      : models/sensor_model_info.json")


## 11. Verifikasi Inferensi

In [ ]:
loaded_model     = tf.keras.models.load_model('models/sensor_autoencoder.keras')
with open('models/sensor_scaler.pkl', 'rb') as f:
    loaded_scaler = pickle.load(f)
loaded_threshold = float(np.load('models/sensor_threshold.npy'))

def infer_from_esp32(temperature, humidity, pressure):
    raw       = np.array([[temperature, humidity, pressure]], dtype=np.float32)
    normed    = loaded_scaler.transform(raw)
    recon     = loaded_model.predict(normed, verbose=0)
    error     = float(np.mean((normed - recon) ** 2))
    return {
        'sensor_error'    : error,
        'sensor_threshold': loaded_threshold,
        'sensor_anomaly'  : error > loaded_threshold
    }

print("=== Test 1: Kondisi Normal ===")
r1 = infer_from_esp32(28.0, 55.0, 1009.0)
print(f"  Sensor error  : {r1['sensor_error']:.6f}")
print(f"  Threshold     : {r1['sensor_threshold']:.6f}")
print(f"  Anomali?      : {'YA ⚠️' if r1['sensor_anomaly'] else 'TIDAK ✅'}")

print()
print("=== Test 2: Suhu Ekstrem ===")
r2 = infer_from_esp32(55.0, 90.0, 1009.0)
print(f"  Sensor error  : {r2['sensor_error']:.6f}")
print(f"  Threshold     : {r2['sensor_threshold']:.6f}")
print(f"  Anomali?      : {'YA ⚠️' if r2['sensor_anomaly'] else 'TIDAK ✅'}")
